# Sales Prediction Based on Advertising Expenditure

**Objective:** predict sales from TV, Radio, and Newspaper advertising expenditure using Multiple Linear Regression. This beginner-friendly notebook covers loading, cleaning, visualization, modeling, evaluation, and new predictions.

The supplied dataset is named `advertising.csv` (lowercase); the loading cell also accepts `Advertising.csv`.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the dataset

In [ ]:
candidates = [Path("Advertising.csv"), Path("advertising.csv")]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Place Advertising.csv (or advertising.csv) in the same folder as this notebook.")

df = pd.read_csv(data_path)
print(f"Loaded {data_path} with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

## 3. Explore and clean the data

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nSummary statistics:")
display(df.describe())

df.columns = df.columns.str.strip()
expected_columns = ["TV", "Radio", "Newspaper", "Sales"]
missing_columns = [column for column in expected_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")
df = df[expected_columns].copy()
for column in expected_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Missing values before cleaning:")
print(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

df = df.dropna().drop_duplicates().reset_index(drop=True)
print(f"\nCleaned dataset shape: {df.shape}")
print("Missing values after cleaning:")
print(df.isna().sum())
df.head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
print("Sales summary:")
display(df["Sales"].describe())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for axis, feature in zip(axes, ["TV", "Radio", "Newspaper"]):
    sns.scatterplot(data=df, x=feature, y="Sales", ax=axis)
    axis.set_title(f"{feature} Advertising vs Sales")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 5. Separate features and target

In [ ]:
# X contains the independent variables; y is the value to predict.
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]
print("Feature columns:", X.columns.tolist())
print("Target column: Sales")

## 6. Split into training and testing sets

In [ ]:
# Keep 20% aside for testing. The seed makes the split repeatable.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Training rows: {len(X_train)} ({len(X_train) / len(X):.0%})")
print(f"Testing rows: {len(X_test)} ({len(X_test) / len(X):.0%})")

## 7. Train a Multiple Linear Regression model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Model intercept:", model.intercept_)
print("Model coefficients:")
for feature, coefficient in zip(X.columns, model.coef_):
    print(f"  {feature}: {coefficient:.4f}")

## 8. Predict and evaluate sales

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Metrics are calculated from this run; no results are hard-coded.
print(f"Mean Absolute Error (MAE): {mae:.3f}")
print(f"Mean Squared Error (MSE): {mse:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
print(f"R-squared (R2): {r2:.3f}")
comparison = pd.DataFrame({"Actual Sales": y_test.values, "Predicted Sales": y_pred})
display(comparison.head(10))

## 9. Plot actual vs predicted sales

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=y_pred)
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], color="red", linestyle="--", label="Perfect prediction")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Predict sales for new advertising inputs

In [ ]:
# Replace these example amounts with your own advertising budgets.
new_advertising = pd.DataFrame({
    "TV": [200.0],
    "Radio": [30.0],
    "Newspaper": [40.0],
})
new_sales_prediction = model.predict(new_advertising)
print("New advertising inputs:")
display(new_advertising)
print(f"Predicted sales: {new_sales_prediction[0]:.2f}")

## Conclusion

This notebook trained Multiple Linear Regression with an 80/20 train/test split. Evaluation metrics and the example prediction are calculated when the notebook runs and depend on the supplied data.